# Compare the Computational Complexity of Matrix Inversion Operations

This notebook contains some functions to profile different matrix inversion operation methods.

### Import libraries

In [1]:
import sys
assert sys.version_info >= (3, 7)

import time
import gc
import psutil
import os
import inspect

import numpy as np

### Data

In [2]:
np.random.seed(123)

In [3]:
process = psutil.Process(os.getpid())

### Helpers for solving

In [4]:
def myInv(size):
    """
    Code provided by Kelsey (2023).
    
    """
    
    intA = np.random.randint(-500,50000, (size,size))
    flA = np.random.rand(size,size)
    A = intA + flA
    b = np.random.randint(-10,100,size)
    return(np.linalg.inv(A))


In [5]:
def myInv_mod(A, b):
    """
    Solve a linear system using matrix inversion.

    This function either inverts a provided matrix `A` or generates a 
    random matrix of given `size` for testing, then returns its inverse.

    Variation of the code provided by Kelsey (2023).

    Args:
        A (np.darray):
            Square matrix to invert. Ignored if `size` is provided.
        b (np.darray):
            Right-hand side vector (not used in the 
            inversion itself but included for uniform interface).

    Returns:
        np.ndarray:
            The matrix solution x = inv(A) @ b.
    
    """
    
    return np.linalg.inv(A) @ b

In [6]:
def Jacobi(A, b, verbose=False):
    """
    Solves a linear system using the Jacobi iterative method, starting from
    a zero vector and updating all components in parallel until convergence.

    Variation of the code provided by Aanjaneya (2017).
    
    Args:
        A (np.ndarray):
            Square coefficient matrix of the linear system.
        b (np.ndarray):
            Right-hand side vector.
        verbose (bool, optional):
            If True, prints intermediate iterates and the iteration count.
    
    Returns:
        np.ndarray:
            Approximate solution vector x to the system A x = b.
    
    """
    
    m = A.shape[0]
    n = A.shape[1]
    if(m!=n):
        print("WARNING: Matrix is not square!")
        return

    # initialize x and x_new
    x = np.zeros(m)
    x_n = np.zeros(m)

    # counter for number of iterations
    iterations = 0

    # perform Jacobi iterations until convergence
    while True:
        for i in range(0,m):
            x_n[i] = b[i]/A[i,i]
            sum = 0
            for j in range(0,m):
                if(j!=i): sum+=A[i,j]*x[j]
            x_n[i] -=sum/A[i,i]

        # stopping criterion
        if(np.linalg.norm(x-x_n,2)<.000001): break

        # copy x_new into x
        for i in range(0,m):
            x[i]=x_n[i]

        # display the updated solution x to inspect the process
        if verbose:
            print("x: ", x)
            iterations+=1
    if verbose:
        print("Iterations: %d", iterations)


In [7]:
def Gauss_Seidel(A, b, verbose=False):
    
    """
    Solves a linear system using the Gauss-Seidel iterative method, starting
    from a zero vector and updating components in-place until convergence.

    Variation of the code provided by Aanjaneya (2017).
    
    Args:
        A (np.ndarray):
            Square coefficient matrix of the linear system.
        b (np.ndarray):
            Right-hand side vector.
        verbose (bool, optional):
            If True, prints intermediate iterates and the iteration count.
    
    Returns:
        np.ndarray:
            Approximate solution vector x to the system A x = b.
    
    """
        
    m = A.shape[0]
    n = A.shape[1]
    if(m!=n):
        print("WARNING: Matrix is not square!")
        return

    # initialize x and x_new
    x = np.zeros(m)
    x_n = np.zeros(m)

    # counter for number of iterations
    iterations = 0

    # perform Gauss-Seidel iterations until convergence
    while True:
        for i in range(0,m):
            x_n[i] = b[i]/A[i,i]
            sum = 0
            for j in range(0,m):
                if(j<i): sum+=A[i,j]*x_n[j]
                if(j>i): sum+=A[i,j]*x[j]
            x_n[i] -=sum/A[i,i]

        # stopping criterion
        if(np.linalg.norm(x-x_n,2)<.000001): break

        # copy x_new into x
        for i in range(0,m):
            x[i]=x_n[i]

        # display the updated solution x to inspect the process
        if verbose:
            print("x: ", x)
            iterations+=1
    if verbose:
        print("Iterations: %d", iterations)


In [8]:
class GaussianElimination:
    """
    Using @staticmethod because each helper works as a standalone, but all
    of them implement row operations for Gaussian Elimination and are
    grouped for clarity and reuse.
    
    This can be turned into using `self`if the input matrix needs to be
    stored in the object for stateful workflows.
    
    Variation of the code provided by Vanderlei (2024).
    
    """
    
    @staticmethod
    def RowSwap(A, k, l):
        """
        Swap two rows of a matrix and return a new matrix.
        
        Args:
            A (np.ndarray):
                Input matrix.
            k (int):
                Index of the first row to swap.
            l (int):
                Index of the second row to swap.
        
        Returns:
            np.ndarray:
                Copy of A with rows k and l swapped.
        
        """
    
        m, n = A.shape
        # Return a new modified matrix without altering the original
        B = np.copy(A).astype("float64")

        for j in range(n):
            temp = B[k, j]
            B[k, j] = B[l, j]
            B[l, j] = temp

        return B

    @staticmethod
    def RowScale(A, k, scale):
        """
        Scale a matrix row by a given factor and return a new matrix.
        
        Args:
            A (np.ndarray):
                Input matrix.
            k (int):
                Index of the row to scale.
            scale (float):
                Scalar multiplier for row k.
        
        Returns:
            np.ndarray:
                Copy of A with row k multiplied by scale.
        
        """
        
        m, n = A.shape
        B = np.copy(A).astype("float64")

        for j in range(n):
            B[k, j] *= scale

        return B

    @staticmethod
    def RowAdd(A, k, l, scale):
        """
        Add a scaled row to another row and return a new matrix.
        
        Args:
            A (np.ndarray):
                Input matrix.
            k (int):
                Index of the source row to be scaled.
            l (int):
                Index of the target row to be updated.
            scale (float):
                Factor multiplying row k before adding to row l.
        
        Returns:
            np.ndarray:
                Copy of A with row l updated as row_l + scale * row_k.
        
        """
        
        
        m, n = A.shape
        B = np.copy(A).astype("float64")

        for j in range(n):
            B[l, j] += B[k, j] * scale

        return B

    @staticmethod
    def GE_solve(A, b):
        """
        Solve a linear system using Gaussian Elimination with partial pivoting.
        
        Performs forward elimination with row operations to obtain an upper
        triangular system, then applies back substitution to compute the
        solution.
        
        Args:
            A (np.ndarray):
                Square coefficient matrix of the linear system.
            b (np.ndarray):
                Right-hand side vector.
        
        Returns:
            np.ndarray:
                Solution vector x to the system A x = b.
                
        """
        
        n = len(b)

        # Forward Elimination
        for k in range(n):
            # Pivot: find max element in column k
            max_row = np.argmax(abs(A[k:, k])) + k
            A = GaussianElimination.RowSwap(A, k, max_row)
            b[k], b[max_row] = b[max_row], b[k]

            # Eliminate rows below pivot
            for i in range(k+1, n):
                if A[k, k] == 0:
                    continue
                factor = A[i, k] / A[k, k]
                A = GaussianElimination.RowAdd(A, k, i, -factor)
                b[i] -= factor * b[k]

        # Back Substitution
        x = np.zeros(n)
        for i in range(n-1, -1, -1):
            x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]

        return x

In [9]:
def my_svd(A, b):
    """
    Perform SVD decomposition.
    
    Args:
            A (np.ndarray):
                Square coefficient matrix of the linear system.
            b (np.ndarray):
                Right-hand side vector.
        
        Returns:
            tuple:
                U (np.ndarray): cols of U are eigenvectors of AA^H
                S (np.ndarray): S has the singular values and nothing else.
                Vh (np.ndarray): rows of V^H are eigenvectors of A^HA
                
    """
    
    U, S, Vh = np.linalg.svd(A, full_matrices=False)

    return U, S, Vh

In [10]:
def my_qr(A, b):
    """
    Perform QR decomposition.

    Args:
            A (np.ndarray):
                Square coefficient matrix of the linear system.
            b (np.ndarray):
                Right-hand side vector. Added for compatibility even if not needed.
        
        Returns:
            tuple:
                Q (np.ndarray):
                    Orthogonal matrix with orthonormal columns such that `Q.T @ Q = I`.
                R (np.ndarray):
                    Upper triangular matrix, where `A = Q @ R` is the QR factorization of matrix A.
    
    """
    
    Q, R = np.linalg.qr(A)

    return Q, R

### Helpers for profiling

In [11]:
def profile_func(func, size_mult=1000, min_range=1, max_range=11):
    """
    Profile a function that takes a single integer size as input.
    
    For increasing matrix sizes, calls func(size), recording the wall-clock
    runtime and additional memory usage for each size.
    
        Args:
            func (callable):
                Function to profile; must accept a single integer size argument.
            size_mult (int, optional):
                Multiplier mapping loop index i to problem size i * size_mult.
            min_range (int, optional):
                Inclusive starting index for the profiling loop.
            max_range (int, optional):
                Exclusive ending index for the profiling loop.
        
        Returns:
            tuple[np.ndarray, np.ndarray]:
                (resTime, resSpace) arrays with per-size runtime and memory usage.
    
    """
    
    n_points = max_range - min_range
    resTime = np.zeros(n_points)
    resSpace = np.zeros(n_points)

    for i in range(min_range, max_range):
        gc.collect()
        baseRam = process.memory_info().rss
        
        start = time.time()
        size = i * size_mult
        func(size)
        ram = process.memory_info().rss
        end = time.time()

        idx = i - min_range
        resTime[idx] = end - start
        resSpace[idx] = ram - baseRam
    
    print(f"resTime = \n {resTime}")
    print(f"resSpace = \n {resSpace}")

    return resTime, resSpace


In [12]:
def profile_func_custom(
    func, A=None, b=None, size_mult=1000, min_range=1, max_range=11):
    """
    Profile a linear solver with custom matrix/vector inputs.
    
    For each problem size, optionally generates a random linear system,
    copies A and b to avoid in-place side effects, and times a single call
    to func(A_copy, b_copy), tracking runtime and memory usage.
    
    Args:
        func (callable):
            Function to profile; must accept (A, b) as arguments.
        A (np.ndarray, optional):
            Coefficient matrix. If None, a random matrix is generated.
        b (np.ndarray, optional):
            Right-hand side vector. If None, a random vector is generated.
        size_mult (int, optional):
            Multiplier mapping loop index i to problem size i * size_mult
            when generating random systems.
        min_range (int, optional):
            Inclusive starting index for the profiling loop.
        max_range (int, optional):
            Exclusive ending index for the profiling loop.
    
    Returns:
        tuple[np.ndarray, np.ndarray]:
            (resTime, resSpace) arrays with per-size runtime and memory usage.
    
    """

    n_points = max_range - min_range
    resTime = np.zeros(n_points)
    resSpace = np.zeros(n_points)

    for i in range(min_range, max_range):
        if A is None or b is None:
            size = i * size_mult
            intA = np.random.randint(-5, 50, (size, size))
            floatA = np.random.rand(size, size)
            A = intA + floatA
            b = np.random.randint(-1, 10, size)

        A_copy = np.copy(A).astype(float)
        b_copy = np.copy(b).astype(float)
        
        gc.collect()
        baseRam = process.memory_info().rss
        
        start = time.time()
        func(A_copy, b_copy)
        ram = process.memory_info().rss
        end = time.time()

        idx = i - min_range
        resTime[idx] = end - start
        resSpace[idx] = ram - baseRam
    
    print(f"resTime = \n{resTime}")
    print(f"\n")
    print(f"resSpace = \n{resSpace}")

    return resTime, resSpace


In [13]:
func_list = [
    myInv_mod,
    Jacobi,
    Gauss_Seidel,
    GaussianElimination.GE_solve,
    my_qr,
    my_svd
]

In [14]:
def runCompare_Funcs(func_list, A=None, b=None, size_mult=1000, min_range=1, max_range=11):
    """
    Run profiling for a list of solver functions.
    
    Iterates over the provided functions, printing a header for each and
    calling `profile_func_custom` with shared size and range parameters.
    
    Args:
        func_list (list[callable]):
            List of solver functions, each accepting (A, b).
        A (np.ndarray, optional):
            Fixed coefficient matrix, reused across all profiled functions.
        b (np.ndarray, optional):
            Fixed right-hand side vector, reused across all profiled functions.
        size_mult (int, optional):
            Multiplier mapping loop index i to problem size i * size_mult.
        min_range (int, optional):
            Inclusive starting index for the profiling loop.
        max_range (int, optional):
            Exclusive ending index for the profiling loop.
    
    Returns:
        None
    
    """
    
    for func in func_list:
        print(f"\nProfiling `{func.__name__}`:")
        print(f"\n") # Create a space below

        # Call profiler with A and b as fixed arguments
        _, _ = profile_func_custom(
            func,
            A=A,
            b=b,
            size_mult=size_mult,
            min_range=min_range,
            max_range=max_range
        )
        

In [19]:
# WARNING: Using `size_mult=1000` may imply a very long runtime
# Moreover, due to numerical instability, Jacobi and Gauss-Seidel may overflow
# Jacobi and Gauss-Seidel are numerically unstable for huge matrices with large entries, 
# especially when the diagonal is small compared to the off-diagonal entries.
runCompare_Funcs(func_list, size_mult=1, min_range=1, max_range=11)


Profiling `myInv_mod`:


resTime = 
[0.00049424 0.00012374 0.00011945 0.00011444 0.00011206 0.00010657
 0.0001049  0.00010586 0.00010586 0.00032282]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `Jacobi`:


resTime = 
[0.00011635 0.00011325 0.0001111  0.00011349 0.00011063 0.00011778
 0.00011301 0.00011516 0.0001061  0.00034785]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `Gauss_Seidel`:


resTime = 
[0.00011802 0.00011373 0.00012279 0.00012279 0.0001204  0.00012231
 0.00017691 0.00012589 0.0003221  0.0002737 ]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `GE_solve`:


resTime = 
[0.00014758 0.00013304 0.00013185 0.00013351 0.00013137 0.00013208
 0.00013614 0.00013423 0.00012994 0.00013542]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `my_qr`:


resTime = 
[0.00018501 0.00017452 0.00017428 0.00017738 0.00017786 0.00018263
 0.00018239 0.00017452 0.00017357 0.00017643]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `my_svd`:


re

The time resources occupied by the different methods are very similar for small matrices, whereas the allocated memory is too small to be returned, but the same similarity is expected. Using bigger matrices can enable the measurement of memory allocation however, this experiment focused on small matrices, since using bigger multiplier `size_mult` causes an overflow on `Jacobi` and `Gauss_Seidel`. This is because, apart from `Jacobi` and `Gauss_Seidel`, which are iterative methods and take `O(n^2*k)`, where `k` is the number of iterations, these operations are `O^3` for matrices with small integer values, but they get harder if great care is not taken with numerical instability. However, it is important to state, that these small differences, will grow as the matrices get bigger.

## Results

Profiling `myInv_mod`:


resTime = 
[0.00049424 0.00012374 0.00011945 0.00011444 0.00011206 0.00010657
 0.0001049  0.00010586 0.00010586 0.00032282]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `Jacobi`:


resTime = 
[0.00011635 0.00011325 0.0001111  0.00011349 0.00011063 0.00011778
 0.00011301 0.00011516 0.0001061  0.00034785]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `Gauss_Seidel`:


resTime = 
[0.00011802 0.00011373 0.00012279 0.00012279 0.0001204  0.00012231
 0.00017691 0.00012589 0.0003221  0.0002737 ]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `GE_solve`:


resTime = 
[0.00014758 0.00013304 0.00013185 0.00013351 0.00013137 0.00013208
 0.00013614 0.00013423 0.00012994 0.00013542]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `my_qr`:


resTime = 
[0.00018501 0.00017452 0.00017428 0.00017738 0.00017786 0.00018263
 0.00018239 0.00017452 0.00017357 0.00017643]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Profiling `my_svd`:


resTime = 
[0.00014305 0.00013852 0.00014997 0.00013828 0.00013971 0.00012708
 0.00013757 0.00013709 0.0004828  0.00014186]


resSpace = 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

REFERENCES:
- Aanjaneya, M. (2017). 4.4. Iterative Methods — CS 323 1.0 documentation. Github.io. https://orionquest.github.io/Numacom/iterative.html
- Kelsey, T. (2023). Topic 2: Matrix inversion and Computational Complexity [Lecture Notebook]. University of St Andrews.
- Vanderlei, B. (2024). Gaussian Elimination — Jupyter Guide to Linear Algebra. Github.io. https://bvanderlei.github.io/jupyter-guide-to-linear-algebra/Gaussian_Elimination.html